# Problem Statement

Large Language Models (LLMs) are typically trained on publicly available datasets and therefore lack access to **private or domain-specific documents**. As a result, when queried about information outside their training distribution, they may generate **inaccurate or hallucinated responses**.

To address this limitation, this project implements a **Retrieval-Augmented Generation (RAG)** pipeline using **LangChain**, enabling LLMs to dynamically access and reason over **external private knowledge sources**.

The system retrieves relevant context from a custom knowledge base and injects it into the LLM prompt, significantly improving accuracy and grounding responses in real data.

---

#  System Architecture: Question Answering Pipeline

The application follows a structured 3-stage pipeline:

##  1. Document Preparation (One-Time per Document)

This stage transforms raw documents into a searchable knowledge base:

- Load input documents (PDF, DOCX, TXT)
- Split documents into smaller, manageable chunks
- Convert text chunks into high-dimensional vector embeddings
- Store embeddings along with metadata in a **Vector Database**

---

##  2. Retrieval Phase (Per Query)

This stage identifies the most relevant information for a given user query:

- Convert the user query into an embedding vector
- Compute similarity scores between query embedding and stored document embeddings
- Rank document chunks based on semantic similarity
- Retrieve top-*k* most relevant chunks

---

##  3. Response Generation (Per Query)

This stage generates the final answer using contextual grounding:

- Combine the user query with retrieved document chunks
- Construct a structured prompt for the LLM
- Generate a context-aware response using the LLM
- Return the final answer to the user

---

#  Tech Stack (OPL Stack)

| Component        | Technology Used |
|----------------|----------------|
| **Vector Database** | Pinecone (scalable) / Chroma (lightweight) |
| **LLM Provider** | OpenAI API |
| **Framework** | LangChain |
| **Frontend/UI** | Streamlit |

---

# Key Features

-  **Multi-format Document Support**  
  Upload and process files in PDF, DOCX, and TXT formats

-  **Customizable Parameters**  
  - Adjustable **chunk size** for document splitting  
  - Configurable **top-k retrieval** for similarity search

-  **Conversational Memory**  
  Maintains chat history to enable context-aware interactions

-  **Dynamic Context Reset**  
  Automatically clears chat memory when:
  - A new document is uploaded  
  - Chunk size is modified  
  - Retrieval parameter (*k*) is changed  

- **Accurate, Context-Grounded Responses**  
  Eliminates hallucinations by anchoring answers in retrieved data

---

#  Core Concept: Retrieval-Augmented Generation (RAG)

This project leverages the RAG paradigm:

> Instead of relying solely on pre-trained knowledge, the model retrieves relevant external information at runtime and uses it to generate accurate, context-aware responses.

---

#  Use Cases

- Enterprise document search
- Research paper Q&A systems
- Internal knowledge base assistants
- Legal / medical document analysis
- Educational assistants for private study material


In [1]:
# Load the env  
import os 
import langchain
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(),override=True)

True

In [2]:
# Fucntion to load data from the pdf 
def load_document(file):
    try:
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading file {file}.....')
        
        loader = PyPDFLoader(file)
        data = loader.load()
        
        return data
    
    except Exception as e:
        print(f"Error loading document: {e}")
        return None

In [3]:
# Test it here 
data=load_document('paper.pdf')
print(data)

Loading file paper.pdf.....


/Users/sheikhfairoz/Desktop/projects/lang_env/lib/python3.11/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


[Document(page_content="Retrieval  Augmented  Generation \n(RAG) Model  \n \n \nAnkit  Mishra  \nDepartment  of Computer  Science  and \nEngineering  \nSharda  University  \nGreater Noida, India \nankit02112002@gmail.com  Aniket  Gupta  \nDepartment  of Computer  Science  and \nEngineering  \nSharda University \nGreater Noida, India \nAniketkg472@gmail.com  (Prof.)Dr. An il Kumar Sagar \nDepartment  of Computer  Science  and \nEngineering  \nSharda University \nGreater Noida, India \nanil.sagar@sharda.ac.in  \n \n \nAbstract —LLM (Large Language Model) is a type of \nArtificial Intelligence algorithm which uses deep learning \ntechniques like neural networks models to generate text -based \nanswers for a variety of user queries. The model gets trained \nover a large number of pre -defined dataset and generate  \nresults based on the facts emerging from this data. LLM RAG \nis a concept where the dataset is continuously fetched from the \nreal time facts rather than pre -stored data. It

In [4]:
print(data[0].page_content)

Retrieval  Augmented  Generation 
(RAG) Model  
 
 
Ankit  Mishra  
Department  of Computer  Science  and 
Engineering  
Sharda  University  
Greater Noida, India 
ankit02112002@gmail.com  Aniket  Gupta  
Department  of Computer  Science  and 
Engineering  
Sharda University 
Greater Noida, India 
Aniketkg472@gmail.com  (Prof.)Dr. An il Kumar Sagar 
Department  of Computer  Science  and 
Engineering  
Sharda University 
Greater Noida, India 
anil.sagar@sharda.ac.in  
 
 
Abstract —LLM (Large Language Model) is a type of 
Artificial Intelligence algorithm which uses deep learning 
techniques like neural networks models to generate text -based 
answers for a variety of user queries. The model gets trained 
over a large number of pre -defined dataset and generate  
results based on the facts emerging from this data. LLM RAG 
is a concept where the dataset is continuously fetched from the 
real time facts rather than pre -stored data. It helps in  
providing the users with most accurate an

In [5]:
print(data[1].metadata)

{'source': 'paper.pdf', 'page': 1}


In [6]:
# Display the number of pages in data 
print(f'The uploaded pdf has {len(data)} pages ')

# Display the number of characters in the page 
print(f'Page 1 has {len(data[0].page_content)} characters in it ')

The uploaded pdf has 6 pages 
Page 1 has 5889 characters in it 


In [7]:
# Function that accepts other types of documents as well

def load_documnets(file):
    """
    This function accepsts files with extenstion '.txt' , '.pdf' and '.doxc'
    and returns data loaded from the file 

    """
    import os 
    name,extension=os.path.splitext(file)
    
    if extension == ".pdf":
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading {file}....')
        loader=PyPDFLoader(file)

    elif extension == '.docx':
        from langchain_community.document_loaders import Docx2txtLoader
        print(f'Loading {file}....')
        loader=Docx2txtLoader(file)

    elif extension=='.txt':
        from langchain_community import TextLoader
        print(f'Loading the {file}......')
        loader=TextLoader(file)

    else:
        print('Document format is not supported for now ')
    
    data=loader.load()
    return data



In [8]:
# Loading the file from the Public and Private Services 
# pip install wikipedia -q

def load_from_wikipedia(query,lang='en',load_max_docs=2):
    """
    This function will load data from the wikipedia

    query = Enter what you want to search in wikipedia
    lang  = Used for fetching response in particular language accepts en for english du for dutch etc

    """
    from langchain_community.document_loaders import WikipediaLoader

    loader=WikipediaLoader(query=query,lang=lang,load_max_docs=load_max_docs)

    data=loader.load()
    return data

In [9]:
# pip install wikipedia

In [10]:
import warnings 
warnings.filterwarnings("ignore")

In [11]:
# Testing the wikipedia function
query="Islamic University of Science and Technology Awantipora"
data=load_from_wikipedia(query=query,lang='en')
print(data[0].page_content)

Islamic University of Science & Technology (IUST) is a state university situated in Awantipora, in the Union Territory of Jammu and Kashmir, India. The university has been set up as a centre for higher learning for the people of the Jammu and Kashmir State and its neighbouring regions. It is recognised by the UGC and AICTE and is a member of AIU.
It was established through an act passed by the Jammu and Kashmir State Legislative Assembly in November 2005. Islamic University is located 25 km from the summer capital of the state, Srinagar.
The chancellor of the university currently is LG Manoj Sinha. The executive council, chaired by the Vice Chancellor, is the executive authority of the university. The strong science and technology curriculum is complemented by a School of Humanities and Social Sciences. The university focuses on career development and overall personality enhancement and tries to ensure education for leadership.
Since the state of Jammu and Kashmir was turned into a Uni

# Chunk the Data 

In [12]:
# Function that is used for Chunking the data 
def chunk_data(data,chunk_size=256):
    """
    This function takes data loaded from lanchain_documentloaders and then chunks them 
    data : Data loaded using lanchain documents loader
    chunk_size : Provide an in integer ranging from 256 to any you like [256,512,1024 ]
    """
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    chunks=text_splitter.split_documents(data)
    return chunks

In [13]:
# Load data using any document loader
data=load_from_wikipedia("Andrew NG")
# chunk it now 
chunks=chunk_data(data,chunk_size=256)
print(f'Obtaiend {len(chunks)} chunks of data ')

Obtaiend 77 chunks of data 


In [14]:
print(chunks[0].page_content)

Andrew Yan-Tak Ng (Chinese: 吳恩達; born April 18, 1976) is a British-American computer scientist and technology entrepreneur focusing on machine learning and artificial intelligence (AI). Ng was a cofounder and head of Google Brain and was the former Chief


# Embed the Chunks and Upload them to a Vector Database

In [15]:
# pip install torch torchvision torchaudio

In [16]:
# pip install transformers==4.41.2

In [17]:

# pip install sentence-transformers==2.7.0

In [18]:
def insert_or_fetch_embedding(index_name, chunks):
    import os
    import time
    from pinecone import Pinecone, ServerlessSpec
    from langchain_pinecone import PineconeVectorStore
    from langchain_community.embeddings import HuggingFaceEmbeddings
    from dotenv import load_dotenv, find_dotenv

    load_dotenv(find_dotenv(), override=True)

    # Pinecone init
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

    #  LOCAL embeddings (NO API, NO ERRORS)
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    existing_indexes = [i.name for i in pc.list_indexes()]

    if index_name in existing_indexes:
        print(f"Index '{index_name}' exists → loading...")

        vector_store = PineconeVectorStore(
            index_name=index_name,
            embedding=embeddings
        )

    else:
        print(f" Creating index '{index_name}'...")

        pc.create_index(
            name=index_name,
            dimension=384,   # correct for MiniLM
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        time.sleep(10)

        print(" Storing embeddings...")

        vector_store = PineconeVectorStore.from_documents(
            documents=chunks,
            embedding=embeddings,
            index_name=index_name
        )

        print(" Index created and data stored")

    return vector_store

In [19]:
# pip install sentence-transformers

In [20]:
# pip install sentence-transformers torch

In [21]:
# Test case
data=load_from_wikipedia("Something about Kashmir",lang='en')
chunks=chunk_data(data)
index_name='askwikipedia'
vector_store=insert_or_fetch_embedding(index_name=index_name,chunks=chunks)

Index 'askwikipedia' exists → loading...


In [22]:
# def delete_pinecone_index():

#     """
#     This function is creaetd to delete all indxes from the pinecone as we are using free tier for now 
#     """
#     import os
#     from pinecone import Pinecone
#     from dotenv import load_dotenv, find_dotenv

#     # Load env
#     load_dotenv(find_dotenv(), override=True)

#     # Init Pinecone
#     pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

#     # Get all indexes
#     indexes = [i.name for i in pc.list_indexes()]

#     if not indexes:
#         print("No indexes found.")
#         return

#     print("Deleting indexes...")

#     for index in indexes:
#         print(f"Deleting: {index}")
#         pc.delete_index(index)

#     print("All indexes deleted successfully.")

In [23]:
# delete_pinecone_index()

# Ask and Get answers Based on similarity search

In [24]:
def ask_and_get_answers(vector_store, q):
    """
    Ask question from vector DB using Gemini (FIXED)
    """

    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain.chains import RetrievalQA
    from dotenv import load_dotenv, find_dotenv

    load_dotenv(find_dotenv(), override=True)

    # add convert_system_message_to_human=True
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",   # safer model
        temperature=0,
        convert_system_message_to_human=True
    )

    #  correct spelling (retriever)
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    #  use proper constructor
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever
    )

    #  correct input format
    result = qa_chain.invoke({"query": q})

    return result["result"]

In [25]:
# Comment this for now
# q = "Kashmir is in which country?"

# answer = ask_and_get_answers(vector_store, q)
# print(answer)

In [26]:
#Comment this out for now
#  # lets loop it now  
# import time
# i=1
# print(f'Write `quit` or `exit`  or `bye` to end Conversation ')

# while True:
#     q=input(f"Enter Question {i} : ")
#     i=i+1
#     if q.lower() in ["exit","bye","quit"]:
#         print('Quitting Bye .....')
#         time.sleep(2)
#         break
#     answer=ask_and_get_answers(vector_store=vector_store,q=q)
#     print(f'\n Answer : {answer}')
#     print(f'\n {"-"*50 } \n ')
    

# Lets now use Chroma as a vector store 
Chroma is an in memory vector store making it better for making small to medium sized projects

In [27]:
# pip install -q chromadb

In [28]:
def create_embedding_chroma(chunks,persist_directory="./chroma_db"):
    """
    This function creates the embeddings 
    uses OpenAI embedding class or we will use something that is free 
    and then save the embeddings
    """

    from langchain.vectorstores import Chroma
    from langchain_community.embeddings import HuggingFaceEmbeddings
    #  LOCAL embeddings (NO API, NO ERRORS)
    embeddings =HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )
    vector_store=Chroma.from_documents(chunks,embedding=embeddings,persist_directory=persist_directory)

    return vector_store

    

In [29]:
def load_embeddings_chroma(persist_directory="./chroma_db"):
    """
    This Function loads the embeddings from disk to a vector store object
    """

    from langchain.vectorstores import Chroma
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings=HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vector_store=Chroma(persist_directory=persist_directory,
                        embedding_function=embeddings)
    
    return vector_store

In [30]:
data=load_documnets('paper.pdf')
chunks=chunk_data(data)
vector_store=create_embedding_chroma(chunks)


Loading paper.pdf....


In [31]:
# Comment this out for now
# q="what is the limit per querry ?"
# answer=ask_and_get_answers(vector_store,q)
# print(answer)

In [32]:
# Comment this out for now
# q="what is the querry to backend"
# answer=ask_and_get_answers(vector_store,q)
# print(answer)

# Lets now test the function that loads the embeddings from disk into a vector store object

In [33]:
# Comment this out for now
# db=load_embeddings_chroma()
# q="LLM RAG models hold the promise of what thing? "
# answer=ask_and_get_answers(vector_store,q)
# print(answer)


In [34]:
# # Comment this out for now
# q="what is the limit per querry ?"
# answer=ask_and_get_answers(vector_store,q)
# print(answer)

In [35]:
# # Comment this out for now
# q="Multiply the number you got with 10?"
# answer=ask_and_get_answers(vector_store,q)
# print(answer)

* >It is clear that it does not have memory it looses the context so let's fix this now 

# ADDING MEMORY TO RAG SYSTEM


In [46]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    convert_system_message_to_human=True
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)
crc = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    chain_type='stuff',
    verbose=True
)

In [47]:
def ask_question(q,chain):
    """
    This function asks for value 'q' and 'chain' and answers in return
    q: Querry or prompt
    chain: chain (Conversational Retrival chain)

    """
    result=chain.invoke({'question':q})
    return result

In [44]:
# Lets now test the memory of the 
data=load_document('paper.pdf')
chunks=chunk_data(data)
vector_store=create_embedding_chroma(chunks)


Loading file paper.pdf.....


In [49]:
q="what is the limit per querry ?"
answer=ask_question(q,crc)
print(answer['answer'])



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: what is the limit per querry ?
Assistant: The limit per query is 50 videos.
Follow Up Input: what is the limit per querry ?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
500 videos as there is a limit to 50 videos per query.  
 
 
Figure  3: Top videos  using  the search  query  
G-CARED 2025 - First Global Conference on AI Research and Emerging Developments
G-CARED 2025  |  DOI: 10.63169/GCARED2025.p16  |  Page 117

500 videos as there is a limit to 50 videos per query.  
 
 
Figu

In [50]:
q="Multiply the limit you got with 10?"
answer=ask_and_get_answers(vector_store,q)
print(answer)

The limit is 50 videos per query. Multiplying this by 10 gives 500 videos.
